# WNBA Draft Fit Predictor — Modeling

This notebook builds a model to predict Rookie Impact Score (RIS)
for the 2026 draft class based on NCAA career stats and team context.

Target variable: RIS (Rookie Impact Score)
Training data: 2019-2025 Round 1 rookies
Prediction: 2026 Round 1 draft class

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Load data
rookies_df = pd.read_csv("../data/processed/rookies_clean.csv")
prospect_features = pd.read_csv("../data/processed/prospect_team_features.csv")

print("Rookies:", rookies_df.shape)
print("Prospects:", prospect_features.shape)

Rookies: (96, 13)
Prospects: (12, 27)


In [2]:
# Check what college data we have for historical rookies
print("Sample rookies:")
print(rookies_df[['player', 'college', 'draft_year', 'draft_pick', 'RIS']].head(20).to_string())

Sample rookies:


KeyError: "['RIS'] not in index"

In [5]:
# Get list of all historical rookies we need NCAA stats for
pd.set_option('display.max_rows', None)
print(rookies_df[['player', 'college', 'draft_year', 'draft_pick']].to_string())

                     player             college  draft_year  draft_pick
0              Jackie Young          Notre Dame        2019           1
1                   AD Durr          Louisville        2019           2
2            Teaira McCowan   Mississippi State        2019           3
3       Katie Lou Samuelson               UConn        2019           4
4          Arike Ogunbowale          Notre Dame        2019           5
5          Napheesa Collier               UConn        2019           6
6              Kalani Brown              Baylor        2019           7
7              Alanna Smith            Stanford        2019           8
8           Kristine Anigwe          California        2019           9
9              Kiara Leslie            NC State        2019          10
10           Brianna Turner          Notre Dame        2019          11
11             Ezi Magbegor                 NaN        2019          12
12        Sophie Cunningham            Missouri        2019     

In [8]:
# Get only NCAA players from our cleaned rookie dataset
ncaa_rookies = rookies_df[rookies_df['college'].notna()].copy()
print(f"NCAA rookies who actually played: {len(ncaa_rookies)}")
pd.set_option('display.max_rows', None)
print(ncaa_rookies[['player', 'college', 'draft_year', 'draft_pick']].to_string())

NCAA rookies who actually played: 86
                     player             college  draft_year  draft_pick
0              Jackie Young          Notre Dame        2019           1
1                   AD Durr          Louisville        2019           2
2            Teaira McCowan   Mississippi State        2019           3
3       Katie Lou Samuelson               UConn        2019           4
4          Arike Ogunbowale          Notre Dame        2019           5
5          Napheesa Collier               UConn        2019           6
6              Kalani Brown              Baylor        2019           7
7              Alanna Smith            Stanford        2019           8
8           Kristine Anigwe          California        2019           9
9              Kiara Leslie            NC State        2019          10
10           Brianna Turner          Notre Dame        2019          11
12        Sophie Cunningham            Missouri        2019          13
14            Chloe Jackson